In [ ]:
"""
Прогнозування волатильності фондового індексу S&P 500
та її застосування в оцінюванні ринкового ризику.

Структура (блоки):
  0. Налаштування та імпорти
  1. Збір даних (S&P 500, VIX, макропоказники FRED)
  2. Очищення даних
  3. Конструювання ознак (повна денна дисперсія: Гарман-Клас + овернайт)
  4. Поділ на навчальну та тестову вибірки (ФІКСОВАНІ дати)
  4.5. Передмодельна діагностика (ADF, Ljung-Box, ARCH-LM)
  4.6. Описова статистика (mean, σ, skew, kurt, Jarque-Bera)
  4.7. ACF дохідностей і квадратів (візуальний доказ кластеризації)
  4.8. Санітарна перевірка екстремальних днів (топ-10 найгірших днів)
  5. GARCH(1,1) та GJR-GARCH з skew-t (rolling) + збереження параметрів
  6. HAR та HAR-X (з VIX) walk-forward + поправка Дуана на смерінг
  7. XGBoost, Random Forest та XGBoost-QLIKE (walk-forward) +
     поправка Дуана + permutation importance
  8. Оцінювання моделей: RMSE, MAE, QLIKE
  8.5. Тести значущості різниці прогнозів: Diebold-Mariano попарно
  9. Value-at-Risk та бек-тестування (Kupiec POF, Christoffersen independence)
  10. Прогнози на горизонті 5 днів (robustness check)
  11. Збереження результатів у Excel
"""

In [ ]:
# ============================================================
# БЛОК 0. Налаштування та імпорти
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from arch import arch_model
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

# --- Параметри дослідження (ВСІ ФІКСОВАНІ для відтворюваності) -----
TICKER          = "^GSPC"              # S&P 500
START_DATE      = "2005-01-01"
END_DATE        = "2026-06-15"         # ФІКСОВАНА межа збору даних
TRAIN_END       = "2024-06-14"         # остання дата навчальної частини
TEST_START      = "2024-06-15"         # перша дата тестової частини
# тестова вибірка: 2024-06-15..2026-06-15  ~ 504 торгових дні
EST_WINDOW      = 1500                 # вікно оцінювання GARCH (rolling)
GARCH_REFIT_EVERY = 5                  # перенавчання GARCH що N днів
HAR_REFIT_EVERY = 22                   # перенавчання HAR щомісяця
ML_REFIT_EVERY  = 22                   # перенавчання ML щомісяця
VAR_LEVELS      = [0.95, 0.99]         # рівні довіри для VaR
H_STEPS         = [1, 5]               # горизонти прогнозу (днів)
RANDOM_STATE    = 42

np.random.seed(RANDOM_STATE)

In [ ]:
# ============================================================
# БЛОК 1. Збір даних
# ============================================================
def fetch_data(ticker=TICKER, start=START_DATE, end=END_DATE):
    """Завантажує денні котирування індексу, VIX та макропоказники FRED."""
    import yfinance as yf
    import time
    import requests

    FRED_API_KEY = "7a6070f9bde9ffe2e12e5e5cb3478e17"

    # 1.1 Котирування індексу (OHLC) -- потрібні Open/High/Low/Close
    print(f"Завантаження котирувань {ticker}...")
    raw = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    px = raw[["Open", "High", "Low", "Close"]].copy()

    # 1.2 Індекс волатильності VIX
    vix = None
    for vix_ticker in ["^VIX", "%5EVIX", "VIX"]:
        try:
            vr = yf.download(vix_ticker, start=start, end=end, auto_adjust=False, progress=False)
            if vr is not None and not vr.empty:
                if isinstance(vr.columns, pd.MultiIndex):
                    vr.columns = vr.columns.get_level_values(0)
                vix = vr["Close"]
                break
        except Exception:
            continue

    # Якщо Yahoo не віддав VIX -- беремо з FRED (VIXCLS)
    if vix is None or vix.dropna().empty:
        try:
            print("Yahoo не повернув VIX. Завантаження VIXCLS через FRED API...")
            url = f"https://api.stlouisfed.org/fred/series/observations?series_id=VIXCLS&api_key={FRED_API_KEY}&file_type=json"
            data = requests.get(url).json()
            v_df = pd.DataFrame(data['observations'])
            v_df['date'] = pd.to_datetime(v_df['date'])
            v_df['value'] = pd.to_numeric(v_df['value'], errors="coerce")
            vix = v_df.set_index("date")["value"]
            print("VIX успішно взято з FRED.")
        except Exception as e:
            print("Попередження: не вдалося завантажити VIX:", e)
            vix = pd.Series(dtype=float)

    px["VIX"] = vix.reindex(px.index)

    # 1.3 Макропоказники з FRED
    fred_series = {"y10": "DGS10", "y3m": "DGS3MO", "credit_spread": "BAMLH0A0HYM2"}
    for col, code in fred_series.items():
        url = f"https://api.stlouisfed.org/fred/series/observations?series_id={code}&api_key={FRED_API_KEY}&file_type=json"
        try:
            print(f"Завантаження {code} ({col}) через FRED API...")
            resp = requests.get(url, timeout=15).json()
            if 'observations' in resp:
                s = pd.DataFrame(resp['observations'])
                s['date'] = pd.to_datetime(s['date'])
                s['value'] = pd.to_numeric(s['value'], errors="coerce")
                s = s.set_index("date")["value"]
                px[col] = s.reindex(px.index)
                print(f"{code}: ok.")
            else:
                print(f"FRED не повернув дані для {code}.")
                px[col] = np.nan
        except Exception as e:
            print(f"Помилка завантаження {code} з FRED:", e)
            px[col] = np.nan
        time.sleep(2)

    px.index = pd.to_datetime(px.index)
    px = px.sort_index()
    return px

In [ ]:
# ============================================================
# БЛОК 2. Очищення даних
# ============================================================
def clean_data(df):
    """
    Очищення:
      - forward-fill макропоказників (різна частота публікації);
      - видалення рядків без ціни закриття;
      - фільтр явних збоїв за дохідністю (|r| > 40% -- майже напевно помилка
        котирування для S&P 500, реальні екстремуми менші).
    Дані вибухонебезпечно великих рухів (2008, 2020, 2025) НЕ видаляються --
    вони є реальним сигналом для волатильності.
    """
    df = df.copy()
    price_cols = ["Open", "High", "Low", "Close"]
    df.loc[(df[price_cols] <= 0).any(axis=1), price_cols] = np.nan
    for c in ["VIX", "y10", "y3m", "credit_spread"]:
        if c in df.columns:
            df[c] = df[c].ffill()
    df = df.dropna(subset=["Close"])
    logret = np.log(df["Close"]).diff()
    bad = logret.abs() > 0.40
    if bad.sum() > 0:
        print(f"Очищення: вилучено {int(bad.sum())} аномальних рядків (|r|>40%).")
        df = df.loc[~bad]
    return df

In [ ]:
# ============================================================
# БЛОК 3. Конструювання ознак
# ============================================================
def realized_variance_full(df):
    """
    ПОВНА денна реалізована дисперсія (close-to-close), узгоджена з дохідністю,
    до якої згодом застосовується VaR. Складається з двох компонент:
      1) внутрішньоденна -- оцінювач Гармана-Класса за O-H-L-C;
      2) овернайт -- (ln(Open_t / Close_{t-1}))^2.
    Без овернайт-компоненти Гарман-Клас систематично занижує повну дисперсію,
    що робить його непорівнянним із close-to-close дохідністю.
    Повертає дисперсію у %^2.
    """
    hl = np.log(df["High"] / df["Low"]) ** 2
    co = np.log(df["Close"] / df["Open"]) ** 2
    intraday = 0.5 * hl - (2 * np.log(2) - 1) * co
    overnight = (np.log(df["Open"] / df["Close"].shift(1))) ** 2
    full = (intraday + overnight).clip(lower=1e-8)
    return full * 1e4                  # переведення у %^2


def build_features(df):
    """
    Цільова змінна + усі предиктори. Усі екзогенні фактори -- з лагом.

    Стратегія dropna (виправлення проблеми обрізання історії):
      - КРИТИЧНІ колонки (без них задачі нема): ret, rv, rv_d, rv_w, rv_m,
        ret_lag1, absret_l1, ret2_l1 -- по них робиться суворий dropna.
        Це обрізає лише ~22 перші дні (через rv_m = 22-денне середнє).
      - ОПЦІОНАЛЬНІ (потрібні тільки для ML): vix_l1, vix_chg, term_spread,
        credit_spread -- НЕ обрізають весь датасет. Можуть мати NaN на початку,
        якщо відповідна серія FRED/Yahoo стартує пізніше за S&P 500.
        ML-модель усередині walk-forward сама дропне рядки з NaN ознаками.
      - FRED-серії публікуються рідше за біржу (свята, паузи) -- ffill всередині
        ряду, але БЕЗ заповнення на початку, де реальних даних ще не існує.

    GARCH і HAR використовують лише критичні колонки -> працюють на повній історії.
    HAR-X і ML починаються з тієї дати, де доступний VIX (зазвичай майже вся історія).
    """
    out = pd.DataFrame(index=df.index)

    # 3.1 Дохідність close-to-close у відсотках
    out["ret"] = 100.0 * np.log(df["Close"]).diff()

    # 3.2 Єдиний таргет: повна денна дисперсія (intraday + overnight)
    out["rv"]   = realized_variance_full(df)
    out["rvol"] = np.sqrt(out["rv"])

    # 3.3 Компоненти HAR (лаги)
    out["rv_d"] = out["rv"].shift(1)
    out["rv_w"] = out["rv"].rolling(5).mean().shift(1)
    out["rv_m"] = out["rv"].rolling(22).mean().shift(1)

    # 3.4 Фактори, похідні від самих дохідностей (теж критичні для ML)
    out["ret_lag1"]  = out["ret"].shift(1)
    out["absret_l1"] = out["ret"].abs().shift(1)
    out["ret2_l1"]   = (out["ret"] ** 2).shift(1)

    # 3.5 Опціональні екзогенні фактори.
    #     ffill ВСЕРЕДИНІ серії (свята, публікаційні паузи).
    #     Але не дозволяємо bfill / fillna на початку, де даних просто нема.
    def _ffill_after_first_valid(s):
        if s is None or s.dropna().empty:
            return s
        first = s.first_valid_index()
        result = s.copy()
        result.loc[first:] = result.loc[first:].ffill()
        return result

    vix_series         = _ffill_after_first_valid(df.get("VIX"))
    y10_series         = _ffill_after_first_valid(df.get("y10"))
    y3m_series         = _ffill_after_first_valid(df.get("y3m"))
    credit_series      = _ffill_after_first_valid(df.get("credit_spread"))

    out["vix_l1"]        = vix_series.shift(1)         if vix_series is not None else np.nan
    out["vix_chg"]       = vix_series.diff().shift(1)  if vix_series is not None else np.nan
    if y10_series is not None and y3m_series is not None:
        out["term_spread"] = (y10_series - y3m_series).shift(1)
    else:
        out["term_spread"] = np.nan
    out["credit_spread"] = credit_series.shift(1)      if credit_series is not None else np.nan

    # 3.6 Діагностика пропусків ДО dropna -- щоб одразу побачити, хто й де "з'їдає" історію
    print("\n=== Діагностика пропусків у feat (до dropna) ===")
    diag = pd.DataFrame({
        "first_valid": [out[c].first_valid_index() for c in out.columns],
        "last_valid":  [out[c].last_valid_index()  for c in out.columns],
        "n_NaN":       out.isna().sum().values,
        "n_total":     [len(out)] * len(out.columns),
    }, index=out.columns)
    print(diag.to_string())

    # 3.7 dropna ТІЛЬКИ по критичних колонках
    MANDATORY = ["ret", "rv", "rv_d", "rv_w", "rv_m",
                 "ret_lag1", "absret_l1", "ret2_l1"]
    n_before = len(out)
    out = out.dropna(subset=MANDATORY)
    n_after = len(out)
    print(f"\ndropna(subset=критичні): {n_before} -> {n_after} рядків "
          f"(обрізано {n_before - n_after} початкових днів через rv_m=22-дн.середнє).")
    print(f"Період feat: {out.index[0].date()} .. {out.index[-1].date()}")

    # 3.8 Перевірка опціональних: якщо колонка повністю NaN -- попередження
    OPTIONAL = ["vix_l1", "vix_chg", "term_spread", "credit_spread"]
    for c in OPTIONAL:
        n_nan = int(out[c].isna().sum())
        if n_nan == len(out):
            print(f"⚠️  Опціональна колонка '{c}' порожня -- буде виключена з ML.")
        elif n_nan > 0:
            first_ok = out[c].first_valid_index()
            print(f"   {c}: {n_nan}/{len(out)} NaN (перший валідний {first_ok.date()}). ОК.")
    return out

In [ ]:
# ============================================================
# БЛОК 4. Поділ (ФІКСОВАНІ дати, без рухомої межі)
# ============================================================
def train_test_split_fixed(feat, train_end=TRAIN_END, test_start=TEST_START):
    """
    Хронологічний поділ за ФІКСОВАНИМИ календарними датами:
      train = ... <= train_end ;   test = >= test_start.
    Жодних "приблизно 2 років" чи "до сьогодні".
    """
    train_end_ts = pd.Timestamp(train_end)
    test_start_ts = pd.Timestamp(test_start)
    train = feat.loc[feat.index <= train_end_ts]
    test  = feat.loc[feat.index >= test_start_ts]
    split = len(train)
    print(f"\nПоділ вибірки (ФІКСОВАНІ дати):")
    print(f"  train: {train.index[0].date()} .. {train.index[-1].date()}  (N = {len(train)})")
    print(f"  test:  {test.index[0].date()} .. {test.index[-1].date()}    (N = {len(test)})")
    return train, test, split


# БЛОК 4.5. Передмодельна діагностика
def diagnostics(ret):
    """ADF + Ljung-Box на r^2 + ARCH-LM (Engle)."""
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

    r = ret.dropna().values
    res = {}

    adf_stat, adf_p, *_ = adfuller(r, autolag="AIC")
    res["ADF"] = {"stat": adf_stat, "pvalue": adf_p}

    lb = acorr_ljungbox(r ** 2, lags=[10], return_df=True)
    res["LjungBox_r2"] = {"stat": float(lb["lb_stat"].iloc[0]),
                          "pvalue": float(lb["lb_pvalue"].iloc[0])}

    arch_stat, arch_p, _, _ = het_arch(r, nlags=10)
    res["ARCH_LM"] = {"stat": arch_stat, "pvalue": arch_p}

    print("\n=== Діагностика ряду дохідності ===")
    print(f"ADF:           stat={adf_stat:8.3f}   p-value={adf_p:.4f}  "
          f"-> {'стаціонарний' if adf_p < 0.05 else 'НЕстаціонарний'}")
    print(f"Ljung-Box r^2: stat={res['LjungBox_r2']['stat']:8.3f}   "
          f"p-value={res['LjungBox_r2']['pvalue']:.4f}  "
          f"-> {'є автокор. квадратів (кластеризація)' if res['LjungBox_r2']['pvalue'] < 0.05 else 'немає автокор.'}")
    print(f"ARCH-LM:       stat={arch_stat:8.3f}   p-value={arch_p:.4f}  "
          f"-> {'є ARCH-ефект -- GARCH виправданий' if arch_p < 0.05 else 'ARCH-ефект відсутній'}")
    return res


# БЛОК 4.6. Описова статистика (для розділу 3.1)
def descriptive_stats(ret):
    """Стандартна описова статистика дохідностей з Jarque-Bera."""
    from scipy.stats import jarque_bera, skew, kurtosis

    r = ret.dropna().values
    jb_stat, jb_p = jarque_bera(r)
    desc = {
        "N":            len(r),
        "Mean":         float(np.mean(r)),
        "Std":          float(np.std(r, ddof=1)),
        "Min":          float(np.min(r)),
        "Max":          float(np.max(r)),
        "Skewness":     float(skew(r)),
        "Kurtosis":     float(kurtosis(r, fisher=False)),   # повний ексцес (3 = normal)
        "ExcessKurt":   float(kurtosis(r, fisher=True)),     # надлишковий ексцес
        "JarqueBera":   float(jb_stat),
        "JB p-value":   float(jb_p),
    }
    print("\n=== Описова статистика денних дохідностей S&P 500, % ===")
    for k, v in desc.items():
        if isinstance(v, int):
            print(f"  {k:14s} = {v}")
        else:
            print(f"  {k:14s} = {v:10.4f}")
    return pd.Series(desc)


# БЛОК 4.7. ACF (доказ кластеризації)
def plot_acf_returns(ret, max_lag=40, fname="fig_acf_returns.png"):
    """ACF самих дохідностей і ACF їх квадратів -- класичний доказ кластеризації."""
    from statsmodels.tsa.stattools import acf

    r = ret.dropna().values
    n = len(r)
    band = 1.96 / np.sqrt(n)                # 95% довірча смуга

    acf_r  = acf(r,        nlags=max_lag, fft=True)[1:]
    acf_r2 = acf(r ** 2,   nlags=max_lag, fft=True)[1:]
    lags = np.arange(1, max_lag + 1)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    for a, vals, ttl in zip(
            ax,
            [acf_r, acf_r2],
            ["ACF дохідностей r_t  (близько 0 -- немає лінійної передбачуваності)",
             "ACF квадратів r_t²  (значущі лаги -- кластеризація волатильності)"]):
        a.bar(lags, vals, color="steelblue", width=0.7)
        a.axhline(0, color="black", lw=0.8)
        a.axhline(band,  color="red", lw=0.8, ls="--")
        a.axhline(-band, color="red", lw=0.8, ls="--")
        a.set_title(ttl, fontsize=10)
        a.set_xlabel("лаг")
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()
    print(f"ACF-графіки збережено: {fname}")


# БЛОК 4.8. Санітарна перевірка екстремальних днів
def sanity_check_extremes(feat, top_n=10):
    """
    Виводить top_n найгірших днів за |дохідністю| та найвищих днів за вол.
    Це коментар-перевірка з боку даних: чи екстремум 2025 року реальний,
    чи артефакт Yahoo (сплити, биті котирування).
    """
    print(f"\n=== Топ-{top_n} найбільших |денних дохідностей| (sanity check) ===")
    worst = feat.assign(abs_ret=feat["ret"].abs()).nlargest(top_n, "abs_ret")
    print(worst[["ret", "rv", "rvol"]].round(3).to_string())
    print(f"\n=== Топ-{top_n} днів за реалізованою волатильністю ===")
    print(feat.nlargest(top_n, "rvol")[["ret", "rv", "rvol"]].round(3).to_string())

In [ ]:
# ============================================================
# БЛОК 5. GARCH(1,1) та GJR-GARCH (rolling) + ЗБЕРЕЖЕННЯ ПАРАМЕТРІВ
# ============================================================
def rolling_garch_forecast(ret, test_index, p=1, q=1, o=0, dist="skewt",
                           est_window=EST_WINDOW, refit_every=GARCH_REFIT_EVERY,
                           var_levels=VAR_LEVELS, horizon=1):
    """
    Прогноз дисперсії на 1 крок уперед методом ковзного вікна.
    o=0 -> GARCH; o=1 -> GJR-GARCH.
    Повертає:
      var_fc      -- Series прогнозованої дисперсії (%^2) на тесті;
      q_fc        -- DataFrame стандартизованих квантилів умовного розподілу;
      params_log  -- список (дата, params) для аналізу динаміки коефіцієнтів;
      last_summary -- текст summary останньої моделі (для розділу результатів).
    """
    ret = ret.dropna()
    all_idx = ret.index
    var_fc = pd.Series(index=test_index, dtype=float)
    alphas = [1 - lvl for lvl in var_levels]
    q_fc = pd.DataFrame(index=test_index,
                        columns=[f"q_{int(l*100)}" for l in var_levels],
                        dtype=float)
    last_res = None
    params_log = []

    for i, t in enumerate(test_index):
        pos = all_idx.get_loc(t)
        start = max(0, pos - est_window)
        history = ret.iloc[start:pos]            # дані ДО t (без зазирання)

        if (last_res is None) or (i % refit_every == 0):
            am = arch_model(history, mean="Constant", vol="GARCH",
                            p=p, o=o, q=q, dist=dist)
            last_res = am.fit(disp="off", show_warning=False)
            res = last_res
            params_log.append((t, last_res.params.to_dict()))
        else:
            am = arch_model(history, mean="Constant", vol="GARCH",
                            p=p, o=o, q=q, dist=dist)
            res = am.fix(last_res.params)

        f = res.forecast(horizon=horizon, reindex=False)
        var_fc.loc[t] = float(f.variance.values[-1, horizon - 1])

        dist_params = res.params[res.model.distribution.parameter_names()] if \
            len(res.model.distribution.parameter_names()) else []
        ppf = res.model.distribution.ppf(alphas, list(dist_params))
        for col, qv in zip(q_fc.columns, np.atleast_1d(ppf)):
            q_fc.loc[t, col] = float(qv)

    # summary останньої моделі (для розділу 3 у тексті)
    try:
        last_summary = str(last_res.summary())
    except Exception:
        last_summary = ""

    return var_fc, q_fc, params_log, last_summary


def extract_garch_param_table(params_log, model_name):
    """Зведена таблиця: останні оцінені параметри GARCH/GJR з усього rolling-перебігу."""
    if not params_log:
        return pd.DataFrame()
    df = pd.DataFrame([dict(date=d, **p) for d, p in params_log])
    df = df.set_index("date")
    print(f"\n=== Параметри {model_name} -- остання оцінка, {df.index[-1].date()} ===")
    print(df.iloc[-1].round(4).to_string())
    print(f"=== Параметри {model_name} -- середнє по rolling-вікнах ===")
    print(df.mean(numeric_only=True).round(4).to_string())
    return df


# Допоміжне: ПОПРАВКА ДУАНА на смерінг (smearing) для log-моделей
def duan_smearing_factor(y_true_log, y_pred_log):
    """
    Поправка Дуана (1983) на ретрансформацію log -> level:
      smearing = mean( exp(residual) )
    Без неї exp(log_pred) систематично занижує сам level через нерівність Єнсена.
    Коефіцієнт обчислюють на in-sample залишках і застосовують до прогнозів:
      sigma2_hat_corrected = smearing * exp(log_pred).
    """
    resid = (y_true_log - y_pred_log).dropna().values
    if len(resid) == 0:
        return 1.0
    return float(np.mean(np.exp(resid)))

In [ ]:
# ============================================================
# БЛОК 6. HAR і HAR-X (walk-forward) + ПАРАМЕТРИ + smearing
# ============================================================
def har_walk_forward(feat, split, refit_every=HAR_REFIT_EVERY,
                     extra_cols=None, model_label="HAR"):
    """
    Базова специфікація HAR:  log(rv_t) = b0 + b_d*log(rv_d) + b_w*log(rv_w) + b_m*log(rv_m).
    extra_cols=['vix_l1'] -> HAR-X (з VIX, на log) -- щоб закрити критику про
    "класична економетрика без екзогенних факторів".
    Walk-forward з розширюваним вікном.
    Поправка Дуана застосовується на in-sample залишках кожного перенавчання.
    Повертає:
      pred_var       -- прогноз дисперсії (%^2);
      coef_summary   -- останні коефіцієнти моделі + R² + AIC;
      last_smearing  -- останній smearing-фактор.
    """
    base_cols = ["rv_d", "rv_w", "rv_m"]
    cols = list(base_cols)
    if extra_cols:
        cols += list(extra_cols)
    X_full = np.log(feat[cols].replace(0, np.nan)).dropna()
    y_full = np.log(feat.loc[X_full.index, "rv"])

    # вирівнюємо індекси після dropna
    aligned_idx = X_full.index
    split_aligned = aligned_idx.get_indexer([feat.index[split]])[0]
    if split_aligned < 0:
        split_aligned = len(aligned_idx) - sum(aligned_idx >= feat.index[split])

    test_aligned = aligned_idx[split_aligned:]
    pred = pd.Series(index=test_aligned, dtype=float)
    last_model = None
    last_smearing = 1.0
    coef_summary = {}

    for i, t in enumerate(test_aligned):
        pos = aligned_idx.get_loc(t)
        if (last_model is None) or (i % refit_every == 0):
            X_train = X_full.iloc[:pos]
            y_train = y_full.iloc[:pos]
            last_model = LinearRegression().fit(X_train, y_train)
            # smearing з in-sample залишків
            y_train_pred = pd.Series(last_model.predict(X_train), index=X_train.index)
            last_smearing = duan_smearing_factor(y_train, y_train_pred)
        pred.loc[t] = last_model.predict(X_full.iloc[[pos]])[0]

    # коефіцієнти останньої моделі + R² + AIC
    X_train = X_full.iloc[:pos]; y_train = y_full.iloc[:pos]
    y_hat = last_model.predict(X_train)
    sse = float(np.sum((y_train - y_hat) ** 2))
    n = len(y_train); k = X_train.shape[1] + 1
    sigma2 = sse / n
    loglik = -n/2 * (np.log(2 * np.pi * sigma2) + 1)
    aic = 2 * k - 2 * loglik
    r2 = 1 - sse / np.sum((y_train - y_train.mean()) ** 2)
    coef_summary = {
        "intercept": float(last_model.intercept_),
        **{c: float(b) for c, b in zip(cols, last_model.coef_)},
        "R2 (in-sample)": float(r2),
        "AIC": float(aic),
        "loglik": float(loglik),
        "smearing": float(last_smearing),
        "N_train": int(n),
    }
    print(f"\n=== {model_label}: остання оцінка коефіцієнтів ===")
    for k_, v in coef_summary.items():
        print(f"  {k_:15s} = {v:.4f}")

    # перевід у дисперсію з поправкою Дуана
    pred_var = last_smearing * np.exp(pred)
    pred_var = pred_var.reindex(feat.index[split:])
    return pred_var, coef_summary, last_smearing

In [ ]:
# ============================================================
# БЛОК 7. ML: XGBoost (звичайний + QLIKE-objective) + RF
# ============================================================
ML_FEATURES = ["rv_d", "rv_w", "rv_m", "ret_lag1", "absret_l1", "ret2_l1",
               "vix_l1", "vix_chg", "term_spread", "credit_spread"]


def tune_xgb(X_train, y_train):
    """Підбір гіперпараметрів XGBoost за TimeSeriesSplit (без зазирання у тест)."""
    from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
    # дропаємо рядки з NaN у фічах (опціональні колонки на початку історії)
    mask = X_train.notna().all(axis=1) & y_train.notna()
    X_train = X_train.loc[mask]
    y_train = y_train.loc[mask]
    print(f"   tune_xgb: після dropna залишилось {len(X_train)} спостережень для CV")
    grid = {
        "n_estimators": [200, 400, 600],
        "max_depth": [2, 3, 4, 5],
        "learning_rate": [0.02, 0.05, 0.1],
        "subsample": [0.8, 1.0],
    }
    base = XGBRegressor(colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1)
    tscv = TimeSeriesSplit(n_splits=4)
    gs = GridSearchCV(base, grid, cv=tscv,
                      scoring="neg_mean_squared_error", n_jobs=-1)
    gs.fit(X_train, y_train)
    print("Найкращі гіперпараметри XGBoost:", gs.best_params_,
          "| CV-RMSE:", round((-gs.best_score_) ** 0.5, 4))
    return gs.best_params_


def tune_rf(X_train, y_train):
    """Підбір гіперпараметрів Random Forest за TimeSeriesSplit."""
    from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
    from sklearn.ensemble import RandomForestRegressor
    mask = X_train.notna().all(axis=1) & y_train.notna()
    X_train = X_train.loc[mask]
    y_train = y_train.loc[mask]
    print(f"   tune_rf: після dropna залишилось {len(X_train)} спостережень для CV")
    grid = {
        "n_estimators": [300, 500],
        "max_depth": [6, 10, None],
        "min_samples_leaf": [10, 20, 50],
    }
    base = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    tscv = TimeSeriesSplit(n_splits=4)
    gs = GridSearchCV(base, grid, cv=tscv,
                      scoring="neg_mean_squared_error", n_jobs=-1)
    gs.fit(X_train, y_train)
    print("Найкращі гіперпараметри Random Forest:", gs.best_params_,
          "| CV-RMSE:", round((-gs.best_score_) ** 0.5, 4))
    return gs.best_params_


# --- кастомна objective для XGBoost: QLIKE на дисперсії, ŷ = log(σ²) ----
def qlike_objective(y_true_log, y_pred_log):
    """
    QLIKE(σ², exp(ŷ)) = σ² exp(-ŷ) + ŷ - log(σ²) - 1.
    Тут y_true_log = log(rv) (бо саме його ми навчаємось передбачати),
    тож σ² = exp(y_true_log). Маємо:
      grad = -exp(y_true_log - y_pred_log) + 1
      hess =  exp(y_true_log - y_pred_log)
    Така objective приводить ML-навчання у відповідність із метрикою,
    за якою судять про моделі. Закриває критику #3.
    """
    yt = np.asarray(y_true_log, dtype=float)
    yp = np.asarray(y_pred_log, dtype=float)
    z = np.exp(yt - yp)
    grad = 1.0 - z
    hess = z
    return grad, hess


def ml_walk_forward(feat, split, kind="xgb", refit_every=ML_REFIT_EVERY,
                    params=None, objective=None, model_label=None):
    """
    Узагальнений walk-forward для ML.
      kind = 'xgb' | 'rf'
      objective = 'qlike' для XGBoost з кастомною QLIKE-цільовою функцією,
                  інакше -- стандартна MSE на log-scale.
    Поправка Дуана застосовується на in-sample залишках КОЖНОГО перенавчання.

    Стійко до NaN у опціональних колонках:
      - колонки, повністю NaN, автоматично виключаються;
      - при навчанні дропаються рядки, де хоч одна з обраних ознак NaN;
      - при прогнозі: якщо рядок t має NaN в X -- прогноз = NaN (буде ffill пізніше).
    """
    from sklearn.ensemble import RandomForestRegressor
    # 1) виключаємо повністю порожні колонки
    feat_cols = [c for c in ML_FEATURES if c in feat.columns and feat[c].notna().any()]
    dropped = [c for c in ML_FEATURES if c not in feat_cols]
    if dropped:
        print(f"   [{model_label or kind}] виключено порожні колонки: {dropped}")
    X = feat[feat_cols]
    y = np.log(feat["rv"])

    test_index = feat.index[split:]
    pred = pd.Series(index=test_index, dtype=float)
    last_model = None
    last_smearing = 1.0
    last_X_train_for_perm = None
    last_y_train_for_perm = None

    def make_model():
        if kind == "rf":
            p = params or dict(n_estimators=400, max_depth=10, min_samples_leaf=20)
            return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **p)
        p = params or dict(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.8)
        if objective == "qlike":
            return XGBRegressor(colsample_bytree=0.8,
                                random_state=RANDOM_STATE, n_jobs=-1,
                                objective=qlike_objective, **p)
        return XGBRegressor(colsample_bytree=0.8,
                            random_state=RANDOM_STATE, n_jobs=-1, **p)

    for i, t in enumerate(test_index):
        pos = feat.index.get_loc(t)
        if (last_model is None) or (i % refit_every == 0):
            # дропаємо рядки з NaN в ознаках на навчальній підвибірці
            X_train_full = X.iloc[:pos]
            y_train_full = y.iloc[:pos]
            mask = X_train_full.notna().all(axis=1) & y_train_full.notna()
            X_train = X_train_full.loc[mask]
            y_train = y_train_full.loc[mask]
            if len(X_train) < 100:
                # замало точок -- залишаємо попередню модель або пропускаємо
                if last_model is None:
                    continue
            else:
                last_model = make_model()
                last_model.fit(X_train, y_train)
                last_X_train_for_perm = X_train
                last_y_train_for_perm = y_train
                # поправка Дуана
                y_train_pred = pd.Series(last_model.predict(X_train), index=X_train.index)
                last_smearing = duan_smearing_factor(y_train, y_train_pred)
        # прогноз -- лише якщо рядок без NaN
        x_t = X.iloc[[pos]]
        if x_t.notna().all(axis=1).iloc[0] and last_model is not None:
            pred.loc[t] = last_model.predict(x_t)[0]
        else:
            pred.loc[t] = np.nan

    # якщо в тестовому шматку є NaN-прогнози через NaN у фічах -- ffill (нечасто)
    n_nan_pred = int(pred.isna().sum())
    if n_nan_pred > 0:
        print(f"   [{model_label or kind}] прогнозів з NaN-фічами: {n_nan_pred}, заповнено ffill")
        pred = pred.ffill().bfill()

    # gain-importance
    importances = pd.Series(last_model.feature_importances_, index=feat_cols)\
                  .sort_values(ascending=False)

    pred_var = last_smearing * np.exp(pred)
    if model_label:
        print(f"   [{model_label}] smearing-фактор Дуана = {last_smearing:.4f}, "
              f"N_train={len(last_X_train_for_perm)}")
    return pred_var, importances, last_smearing, last_model, last_X_train_for_perm, last_y_train_for_perm


def permutation_importance_xgb(model, X_train, y_train, n_repeats=10):
    """
    Permutation importance: пермутуємо кожну колонку n_repeats разів і дивимось
    зниження R² (на in-sample, але результат корисний для аналізу впливу).
    Дає більш надійну картину, ніж сирі gain-importances на корельованих ознаках.
    """
    from sklearn.metrics import r2_score
    rng = np.random.RandomState(RANDOM_STATE)
    base_pred = model.predict(X_train)
    base_score = r2_score(y_train, base_pred)
    rows = []
    for col in X_train.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X_train.copy()
            Xp[col] = rng.permutation(Xp[col].values)
            sc = r2_score(y_train, model.predict(Xp))
            drops.append(base_score - sc)
        rows.append((col, float(np.mean(drops)), float(np.std(drops))))
    out = pd.DataFrame(rows, columns=["Фактор", "Δ R² mean", "Δ R² std"])
    out = out.sort_values("Δ R² mean", ascending=False).reset_index(drop=True)
    print("\n=== Permutation importance (XGBoost, in-sample) ===")
    print(out.round(4).to_string(index=False))
    return out

In [ ]:
# ============================================================
# БЛОК 8. Оцінювання моделей
# ============================================================
def qlike(realized_var, forecast_var):
    """QLIKE-loss (Patton 2011). Робастний до шумності проксі волатильності."""
    f = np.maximum(forecast_var, 1e-8)
    r = np.maximum(realized_var, 1e-8)
    return np.mean(r / f - np.log(r / f) - 1)


def evaluate(realized_var, forecasts: dict):
    """RMSE, MAE на волатильності + QLIKE на дисперсії."""
    rows = []
    rvol_true = np.sqrt(realized_var)
    for name, fvar in forecasts.items():
        fvar = fvar.reindex(realized_var.index)
        fvol = np.sqrt(fvar.clip(lower=1e-8))
        rmse = np.sqrt(np.mean((fvol - rvol_true) ** 2))
        mae  = np.mean(np.abs(fvol - rvol_true))
        ql   = qlike(realized_var.values, fvar.values)
        rows.append({"Модель": name, "RMSE (vol)": rmse, "MAE (vol)": mae, "QLIKE": ql})
    return pd.DataFrame(rows).set_index("Модель").sort_values("QLIKE")


# БЛОК 8.5. Diebold-Mariano (попарне порівняння прогнозів)
def dm_test(loss1, loss2, h=1):
    """
    Тест Diebold-Mariano (1995) з поправкою Harvey-Leybourne-Newbold (1997).
      H0: середня різниця функцій втрат = 0 (моделі рівноцінні).
    loss1, loss2 -- масиви поточасних функцій втрат (наприклад, QLIKE_t).
    h -- горизонт прогнозу.
    Повертає (DM-stat, p-value).
    """
    d = np.asarray(loss1) - np.asarray(loss2)
    d = d[np.isfinite(d)]
    n = len(d)
    if n < 5:
        return np.nan, np.nan
    mean_d = float(np.mean(d))
    # long-run variance: автоковаріації до лагу h-1
    var_d = float(np.var(d, ddof=1))
    for k in range(1, h):
        gamma_k = float(np.mean((d[k:] - mean_d) * (d[:-k] - mean_d)))
        var_d += 2 * gamma_k
    var_d = max(var_d, 1e-12)
    dm = mean_d / np.sqrt(var_d / n)
    # HLN small-sample correction
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm_hln = dm * hln
    # двосторонній p-value за t(n-1)
    pval = 2 * (1 - stats.t.cdf(abs(dm_hln), df=n - 1))
    return float(dm_hln), float(pval)


def qlike_pointwise(realized_var, forecast_var):
    """Поточасний QLIKE-loss (для DM-тесту)."""
    f = np.maximum(np.asarray(forecast_var, dtype=float), 1e-8)
    r = np.maximum(np.asarray(realized_var,  dtype=float), 1e-8)
    return r / f - np.log(r / f) - 1


def dm_matrix(realized_var, forecasts: dict, h=1):
    """
    Матриця попарних DM-тестів на QLIKE-loss.
    У комірці (i, j) -- p-value тесту H0: модель_i = модель_j.
    Знак DM-stat у нижньому трикутнику показує, у чию користь різниця:
      DM < 0 -> i краща (менші втрати); DM > 0 -> j краща.
    """
    names = list(forecasts.keys())
    losses = {n: qlike_pointwise(realized_var, forecasts[n].reindex(realized_var.index).values)
              for n in names}
    p_mat = pd.DataFrame(np.nan, index=names, columns=names)
    s_mat = pd.DataFrame(np.nan, index=names, columns=names)
    for i, ni in enumerate(names):
        for j, nj in enumerate(names):
            if i == j:
                continue
            stat, pv = dm_test(losses[ni], losses[nj], h=h)
            p_mat.loc[ni, nj] = pv
            s_mat.loc[ni, nj] = stat
    print(f"\n=== Diebold-Mariano (QLIKE-loss, h={h}): p-values ===")
    print(p_mat.round(4).to_string())
    print(f"\n=== Diebold-Mariano (QLIKE-loss, h={h}): test statistics ===")
    print("(від'ємний DM-stat: модель рядка краща за модель стовпця)")
    print(s_mat.round(3).to_string())
    return p_mat, s_mat

In [ ]:
# ============================================================
# БЛОК 9. Value-at-Risk та бек-тестування
# ============================================================
def var_from_quantile(cond_vol, std_quantile):
    """1-денний VaR = -(σ_t · q_α), q_α -- стандартизований квантиль (від'ємний)."""
    if isinstance(std_quantile, pd.Series):
        std_quantile = std_quantile.reindex(cond_vol.index)
    return -(cond_vol * std_quantile)


def fit_skewt_quantiles(actual_ret, cond_vol, var_levels=VAR_LEVELS):
    """
    Для моделей без вбудованого розподілу (HAR, HAR-X, XGB, RF) підганяємо
    skew-t до z = r/σ і отримуємо стандартизовані квантилі.
    """
    from arch.univariate import SkewStudent
    z = (actual_ret / cond_vol.reindex(actual_ret.index)).dropna()
    z = z[np.isfinite(z)].values
    alphas = [1 - lvl for lvl in var_levels]
    dist = SkewStudent()
    try:
        from scipy.optimize import minimize
        def negll(params):
            eta, lam = params
            if eta <= 2.05 or not (-0.99 < lam < 0.99):
                return 1e10
            return -np.sum(dist.loglikelihood([eta, lam], z, np.ones_like(z)))
        opt = minimize(negll, x0=[8.0, -0.1], method="Nelder-Mead")
        params = list(opt.x)
        q = dist.ppf(alphas, params)
    except Exception:
        nu_hat, _, _ = stats.t.fit(z, floc=0, fscale=1)
        nu_hat = float(np.clip(nu_hat, 2.1, 200))
        q = stats.t.ppf(alphas, nu_hat) * np.sqrt((nu_hat - 2) / nu_hat)
    return dict(zip([f"q_{int(l*100)}" for l in var_levels], np.atleast_1d(q)))


def kupiec_pof(violations, n, p):
    """Тест Купіца на безумовне покриття."""
    x = int(violations)
    if x == 0:
        return np.nan, np.nan
    pi = x / n
    lr = -2 * (np.log((1 - p) ** (n - x) * p ** x) -
               np.log((1 - pi) ** (n - x) * pi ** x))
    pval = 1 - stats.chi2.cdf(lr, df=1)
    return lr, pval


def christoffersen_independence(hits):
    """Тест Кристофферсена на незалежність порушень."""
    hits = np.asarray(hits).astype(int)
    n00 = n01 = n10 = n11 = 0
    for prev, cur in zip(hits[:-1], hits[1:]):
        if prev == 0 and cur == 0: n00 += 1
        elif prev == 0 and cur == 1: n01 += 1
        elif prev == 1 and cur == 0: n10 += 1
        else: n11 += 1
    if (n01 + n11) == 0 or (n00 + n01) == 0 or (n10 + n11) == 0:
        return np.nan, np.nan
    pi01 = n01 / (n00 + n01)
    pi11 = n11 / (n10 + n11)
    pi   = (n01 + n11) / (n00 + n01 + n10 + n11)
    num = (1 - pi) ** (n00 + n10) * pi ** (n01 + n11)
    den = (1 - pi01) ** n00 * pi01 ** n01 * (1 - pi11) ** n10 * pi11 ** n11
    lr = -2 * np.log(num / den)
    pval = 1 - stats.chi2.cdf(lr, df=1)
    return lr, pval


def var_backtest(actual_ret, cond_vol, quantiles, levels=VAR_LEVELS, model_name=""):
    rows = []
    n = len(actual_ret)
    cv = cond_vol.reindex(actual_ret.index)
    for lvl in levels:
        q = quantiles[f"q_{int(lvl*100)}"]
        var = var_from_quantile(cv, q)
        hits = (actual_ret < -var).astype(int)
        x = int(hits.sum())
        p = 1 - lvl
        lr_uc, pv_uc = kupiec_pof(x, n, p)
        lr_ind, pv_ind = christoffersen_independence(hits.values)
        rows.append({
            "Модель": model_name,
            "Рівень VaR": f"{int(lvl*100)}%",
            "Очікувано порушень": round(p * n, 1),
            "Фактично порушень": x,
            "Частка порушень": round(x / n, 4),
            "Kupiec p-value": round(pv_uc, 3) if pv_uc == pv_uc else None,
            "Christoffersen p-value": round(pv_ind, 3) if pv_ind == pv_ind else None,
        })
    return pd.DataFrame(rows)


def backtest_all_models(actual_ret, model_vols, garch_quantiles):
    frames = []
    for name, vol in model_vols.items():
        if name in garch_quantiles:
            qdf = garch_quantiles[name]
            quantiles = {c: qdf[c] for c in qdf.columns}
        else:
            quantiles = fit_skewt_quantiles(actual_ret, vol)
        frames.append(var_backtest(actual_ret, vol, quantiles, model_name=name))
    return pd.concat(frames, ignore_index=True)

In [ ]:
# ============================================================
# Графіки
# ============================================================
def plot_forecasts(realized_var, forecasts, test_index, fname="fig_forecasts.png"):
    rvol_true = np.sqrt(realized_var)
    plt.figure(figsize=(13, 5.5))
    plt.plot(test_index, rvol_true, color="black", lw=1.2, label="Реалізована вол.")
    palette = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b", "#e377c2"]
    for (name, fvar), col in zip(forecasts.items(), palette):
        plt.plot(test_index, np.sqrt(fvar.reindex(test_index).clip(lower=1e-8)),
                 lw=1.2, alpha=0.85, color=col, label=name)
    plt.title("Прогноз волатильності S&P 500 (out-of-sample)")
    plt.ylabel("Денна волатильність, %")
    plt.legend(loc="upper left", ncol=2, fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()


def plot_metrics(metrics, fname="fig_metrics.png"):
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for i, col in enumerate(["RMSE (vol)", "MAE (vol)", "QLIKE"]):
        metrics[col].plot(kind="bar", ax=ax[i], color="steelblue", edgecolor="black")
        ax[i].set_title(col)
        ax[i].set_xlabel("")
        ax[i].grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()


def plot_importances(importances, perm_importances, fname="fig_importances.png"):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    importances.sort_values().plot(kind="barh", ax=ax[0], color="darkorange",
                                   edgecolor="black")
    ax[0].set_title("Gain-importance (XGBoost)")
    ax[0].grid(alpha=0.3, axis="x")

    pi = perm_importances.set_index("Фактор")["Δ R² mean"].sort_values()
    pi.plot(kind="barh", ax=ax[1], color="darkgreen", edgecolor="black")
    ax[1].set_title("Permutation importance (Δ R²)")
    ax[1].grid(alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()


def plot_var(actual_ret, cond_vol, level, std_quantile, fname="fig_var.png"):
    var = var_from_quantile(cond_vol.reindex(actual_ret.index), std_quantile)
    hits = actual_ret < -var
    plt.figure(figsize=(13, 5.5))
    plt.plot(actual_ret.index, actual_ret, color="gray", lw=0.8, label="Дохідність, %")
    plt.plot(var.index, -var, color="red", lw=1.4, label=f"VaR {int(level*100)}%")
    plt.scatter(actual_ret.index[hits], actual_ret[hits], color="black", s=22,
                zorder=5, label="Порушення")
    plt.title(f"Бек-тестування VaR {int(level*100)}% (S&P 500)")
    plt.ylabel("Дохідність / Negative VaR, %")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()

In [ ]:
# ============================================================
# БЛОК 10. Головна процедура
# ============================================================
def main():
    print(">> Блок 1-2: збір та очищення даних...")
    raw = fetch_data()
    df = clean_data(raw)

    print(">> Блок 3: конструювання ознак (повна денна дисперсія)...")
    feat = build_features(df)

    print(">> Блок 4: ФІКСОВАНИЙ хронологічний поділ...")
    train, test, split = train_test_split_fixed(feat)
    test_index   = feat.index[split:]
    realized_var = feat["rv"].iloc[split:]
    actual_ret   = feat["ret"].iloc[split:]

    print(">> Блок 4.5: передмодельна діагностика...")
    diag = diagnostics(feat["ret"])

    print(">> Блок 4.6: описова статистика...")
    desc = descriptive_stats(feat["ret"])

    print(">> Блок 4.7: ACF дохідностей і квадратів (фіксує кластеризацію)...")
    plot_acf_returns(feat["ret"])

    print(">> Блок 4.8: санітарна перевірка топ-10 екстремумів...")
    sanity_check_extremes(feat, top_n=10)

    print(">> Блок 5: GARCH(1,1) та GJR-GARCH (skew-t, rolling) + параметри...")
    garch_var, garch_q, garch_params, garch_summary = rolling_garch_forecast(
        feat["ret"], test_index, o=0, dist="skewt")
    gjr_var,   gjr_q,   gjr_params,   gjr_summary   = rolling_garch_forecast(
        feat["ret"], test_index, o=1, dist="skewt")
    garch_param_table = extract_garch_param_table(garch_params, "GARCH(1,1)")
    gjr_param_table   = extract_garch_param_table(gjr_params,   "GJR-GARCH")
    print("\n=== Summary останньої моделі GJR-GARCH ===")
    print(gjr_summary)

    print(">> Блок 6: HAR та HAR-X (з VIX) + smearing Дуана...")
    har_var,   har_coefs,   _ = har_walk_forward(feat, split, model_label="HAR")
    harx_var,  harx_coefs,  _ = har_walk_forward(feat, split,
                                                 extra_cols=["vix_l1"],
                                                 model_label="HAR-X (з VIX)")

    print(">> Блок 7: ML -- XGBoost + RF + XGBoost-QLIKE...")
    feat_cols = [c for c in ML_FEATURES if feat[c].notna().any()]
    Xtr = feat[feat_cols].iloc[:split]
    ytr = np.log(feat["rv"]).iloc[:split]
    best_xgb = tune_xgb(Xtr, ytr)
    best_rf  = tune_rf(Xtr, ytr)

    xgb_var,  xgb_imp,  xgb_sm, xgb_model, Xall, yall = ml_walk_forward(
        feat, split, kind="xgb", params=best_xgb, model_label="XGBoost (MSE)")
    rf_var,   rf_imp,   _, _, _, _ = ml_walk_forward(
        feat, split, kind="rf",  params=best_rf,  model_label="Random Forest")
    xgbq_var, xgbq_imp, _, _, _, _ = ml_walk_forward(
        feat, split, kind="xgb", params=best_xgb, objective="qlike",
        model_label="XGBoost-QLIKE (custom obj.)")

    perm_imp = permutation_importance_xgb(xgb_model, Xtr, ytr, n_repeats=10)

    forecasts = {
        "GARCH(1,1)":    garch_var,
        "GJR-GARCH":     gjr_var,
        "HAR":           har_var,
        "HAR-X":         harx_var,
        "XGBoost":       xgb_var,
        "XGBoost-QLIKE": xgbq_var,
        "Random Forest": rf_var,
    }

    print(">> Блок 8: оцінювання моделей (RMSE/MAE/QLIKE)...")
    metrics = evaluate(realized_var, forecasts)
    print("\n=== Порівняння моделей (менше = краще) ===")
    print(metrics.round(4))
    best_model = metrics["QLIKE"].idxmin()
    print(f"\n>> Найкраща модель за QLIKE: {best_model}")

    print(">> Блок 8.5: Diebold-Mariano (попарні p-values на QLIKE-loss)...")
    dm_p, dm_s = dm_matrix(realized_var, forecasts, h=1)

    plot_forecasts(realized_var, forecasts, test_index)
    plot_metrics(metrics)
    plot_importances(xgb_imp, perm_imp)

    print(">> Блок 9: VaR + бек-тестування ВСІХ моделей...")
    model_vols = {name: np.sqrt(fvar.clip(lower=1e-8)) for name, fvar in forecasts.items()}
    garch_q_map = {"GARCH(1,1)": garch_q, "GJR-GARCH": gjr_q}
    bt_all = backtest_all_models(actual_ret, model_vols, garch_q_map)
    print("\n=== Бек-тестування VaR (усі моделі, skew-t розподіл) ===")
    print(bt_all.to_string(index=False))

    best_vol = model_vols[best_model]
    if best_model in garch_q_map:
        best_q = garch_q_map[best_model]["q_99"]
    else:
        best_q = fit_skewt_quantiles(actual_ret, best_vol)["q_99"]
    plot_var(actual_ret, best_vol, 0.99, best_q, fname="fig_var.png")

In [ ]:
    # --- БЛОК 10. Прогнози на горизонті 5 днів (robustness) ----------------
    print(">> Блок 10: прогнози на горизонті 5 днів (robustness check)...")
    garch_var_h5, _, _, _ = rolling_garch_forecast(
        feat["ret"], test_index, o=0, dist="skewt", horizon=5)
    gjr_var_h5,   _, _, _ = rolling_garch_forecast(
        feat["ret"], test_index, o=1, dist="skewt", horizon=5)
    # для HAR/ML "горизонт 5" моделюємо через скорочений тест: rolling-середнє
    # фактичної реалізованої дисперсії за наступні 5 днів проти прогнозу
    rv_h5 = feat["rv"].rolling(5).mean().shift(-4).reindex(test_index).dropna()
    fcasts_h5 = {
        "GARCH(1,1)": garch_var_h5.reindex(rv_h5.index),
        "GJR-GARCH":  gjr_var_h5.reindex(rv_h5.index),
        "HAR":        har_var.rolling(5).mean().shift(-4).reindex(rv_h5.index),
        "HAR-X":      harx_var.rolling(5).mean().shift(-4).reindex(rv_h5.index),
        "XGBoost":    xgb_var.rolling(5).mean().shift(-4).reindex(rv_h5.index),
        "XGBoost-QLIKE": xgbq_var.rolling(5).mean().shift(-4).reindex(rv_h5.index),
        "Random Forest": rf_var.rolling(5).mean().shift(-4).reindex(rv_h5.index),
    }
    metrics_h5 = evaluate(rv_h5, fcasts_h5)
    print("\n=== Порівняння моделей на ГОРИЗОНТІ 5 ДНІВ (robustness) ===")
    print(metrics_h5.round(4))

In [ ]:
    # --- БЛОК 11. Збереження результатів -----------------------------------
    print(">> Блок 11: збереження результатів...")
    out_path = "results.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as xls:
        metrics.to_excel(xls, sheet_name="Метрики_h1")
        metrics_h5.to_excel(xls, sheet_name="Метрики_h5")
        bt_all.to_excel(xls, sheet_name="VaR_бектест", index=False)
        xgb_imp.to_frame("Gain-importance").to_excel(xls, sheet_name="Важливість_XGB_gain")
        perm_imp.to_excel(xls, sheet_name="Важливість_XGB_perm", index=False)
        dm_p.to_excel(xls, sheet_name="DM_pvalues")
        dm_s.to_excel(xls, sheet_name="DM_statistics")
        if garch_param_table is not None and len(garch_param_table):
            garch_param_table.tail(50).to_excel(xls, sheet_name="GARCH_params_tail50")
        if gjr_param_table is not None and len(gjr_param_table):
            gjr_param_table.tail(50).to_excel(xls, sheet_name="GJR_params_tail50")
        pd.Series(har_coefs).to_excel(xls, sheet_name="HAR_coefs")
        pd.Series(harx_coefs).to_excel(xls, sheet_name="HAR-X_coefs")
        desc.to_excel(xls, sheet_name="Описова_статистика")
        pd.DataFrame([
            {"Найкраща модель за QLIKE (h=1)": best_model,
             "Train end": TRAIN_END, "Test start": TEST_START,
             "End date": END_DATE, "Smearing (Duan) увімкнено": True,
             "DM-тести виконано": True}
        ]).to_excel(xls, sheet_name="Підсумок", index=False)
    print(f"Результати збережено: {out_path}")
    print("Готово.")

In [3]:
if __name__ == "__main__":
    main()

>> Блок 1-2: збір та очищення даних...
Завантаження котирувань ^GSPC...
Завантаження DGS10 (y10) через FRED API...
DGS10: ok.
Завантаження DGS3MO (y3m) через FRED API...
DGS3MO: ok.
Завантаження BAMLH0A0HYM2 (credit_spread) через FRED API...
BAMLH0A0HYM2: ok.
>> Блок 3: конструювання ознак (повна денна дисперсія)...

=== Діагностика пропусків у feat (до dropna) ===
              first_valid last_valid  n_NaN  n_total
ret            2005-01-04 2026-06-12      1     5395
rv             2005-01-04 2026-06-12      1     5395
rvol           2005-01-04 2026-06-12      1     5395
rv_d           2005-01-05 2026-06-12      2     5395
rv_w           2005-01-11 2026-06-12      6     5395
rv_m           2005-02-04 2026-06-12     23     5395
ret_lag1       2005-01-05 2026-06-12      2     5395
absret_l1      2005-01-05 2026-06-12      2     5395
ret2_l1        2005-01-05 2026-06-12      2     5395
vix_l1         2005-01-04 2026-06-12      1     5395
vix_chg        2005-01-05 2026-06-12      2     5